# `long` / `wide` — Reference

`long` melts all numeric columns to rows. `wide` pivots a column's values into headers.

| Method | Key params | Purpose |
|---|---|---|
| `.long(col='variable', value='value')` | `col`, `value` | wide → long (melt) |
| `.wide(col='variable', value='value', aggfunc=None)` | `col`, `value`, `aggfunc` | long → wide (pivot) |

They are designed to work together as a pair, but each is also useful standalone.

---

In [32]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import pytae as pt

penguins = pt.sample_data['penguins']
healthexp = pt.sample_data['healthexp']
flights = pt.sample_data['flights']

In [33]:
penguins

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,None
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,NaN,NaN,NaN,NaN,None
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female


## `long` — melt all numeric columns to rows

In [34]:
# Melt all numeric columns to rows — default col/value names
penguins.long().sample(10)


,species,island,sex,variable,value
1250,Chinstrap,Dream,Male,body_mass_g,4100.0
937,Gentoo,Biscoe,Male,flipper_length_mm,220.0
488,Adelie,Dream,Female,bill_depth_mm,16.8
816,Adelie,Torgersen,Female,flipper_length_mm,191.0
1332,Gentoo,Biscoe,Female,body_mass_g,4625.0
324,Gentoo,Biscoe,None,bill_length_mm,47.3
1267,Gentoo,Biscoe,Male,body_mass_g,5850.0
597,Gentoo,Biscoe,Male,bill_depth_mm,17.0
525,Chinstrap,Dream,Male,bill_depth_mm,20.0
997,Gentoo,Biscoe,Male,flipper_length_mm,230.0


In [35]:
# Custom column names for the variable and value columns
penguins.long(col='metric', value='reading').sample(10)


,species,island,sex,metric,reading
72,Adelie,Torgersen,Female,bill_length_mm,39.6
977,Gentoo,Biscoe,Male,flipper_length_mm,223.0
830,Adelie,Dream,Female,flipper_length_mm,188.0
183,Chinstrap,Dream,Male,bill_length_mm,54.2
81,Adelie,Torgersen,Male,bill_length_mm,42.9
45,Adelie,Dream,Male,bill_length_mm,39.6
974,Gentoo,Biscoe,None,flipper_length_mm,214.0
948,Gentoo,Biscoe,Female,flipper_length_mm,208.0
188,Chinstrap,Dream,Female,bill_length_mm,47.6
66,Adelie,Biscoe,Female,bill_length_mm,35.5


In [36]:
# Only non numeric columns survive as id columns — numerics all get melted
# select narrows which numerics get melted
(penguins
 .select(['species', 'island', 'bill_length_mm', 'bill_depth_mm'])
 .long(col='metric', value='reading')
 .sample(12)
)


,species,island,metric,reading
54,Adelie,Biscoe,bill_length_mm,34.5
561,Chinstrap,Dream,bill_depth_mm,18.2
48,Adelie,Dream,bill_length_mm,36.0
380,Adelie,Dream,bill_depth_mm,20.0
270,Gentoo,Biscoe,bill_length_mm,46.6
626,Gentoo,Biscoe,bill_depth_mm,13.9
36,Adelie,Dream,bill_length_mm,38.8
430,Adelie,Dream,bill_depth_mm,19.5
550,Chinstrap,Dream,bill_depth_mm,17.3
218,Chinstrap,Dream,bill_length_mm,50.8


## `wide` — pivot a column's values into headers

In [37]:
# flights has one row per year+month — no aggfunc needed
flights.wide(col='month', value='passengers').head()


,year,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
0,1949,112,118,132,129,121,135,148,148,136,119,104,118
1,1950,115,126,141,135,125,149,170,170,158,133,114,140
2,1951,145,150,178,163,172,178,199,199,184,162,146,166
3,1952,171,180,193,181,183,218,230,242,209,191,172,194
4,1953,196,196,236,235,229,243,264,272,237,211,180,201


In [38]:
# Select only the columns needed so island becomes the clean index
(penguins
 .select(['island', 'species', 'body_mass_g'])
 .wide(col='species', value='body_mass_g', aggfunc='mean')
)


,island,Adelie,Chinstrap,Gentoo
0,Biscoe,3709.659091,NaN,5076.01626
1,Dream,3688.392857,3733.088235,NaN
2,Torgersen,3706.372549,NaN,NaN


## Round-trip: `long` → transform → `wide`
A common pattern: go long to apply uniform operations across all metrics, then come back wide.

In [39]:
# Select only needed columns → long → wide: yields mean bill metrics per island
(penguins
 .select(['island', 'bill_length_mm', 'bill_depth_mm'])
 .long(col='metric', value='reading')
 .wide(col='metric', value='reading', aggfunc='mean')
)


,island,bill_depth_mm,bill_length_mm
0,Biscoe,15.874850,45.257485
1,Dream,18.344355,44.167742
2,Torgersen,18.429412,38.950980
